# CSE 144 Final Project

## Group Members and Task Distribution
- TODO: Fill in group members and tasks

In [1]:
import csv
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from torchvision import datasets, models, transforms

import os

In [2]:
DATA_DIR = os.path.join(os.getcwd(), "ucsc-cse-144-winter-2026-final-project")

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

NUM_CLASSES = 100

## The Network Backbone

We use a pretrained **ResNet-50** as the backbone, replacing the final fully-connected layer to output 100 classes. The pretrained ImageNet weights provide a strong feature extractor for our small dataset.

In [3]:
def get_model(num_classes=NUM_CLASSES):
    # --- EfficientNet-B0 (default) ---
    # model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    # in_features = model.classifier[1].in_features
    # model.classifier[1] = nn.Linear(in_features, num_classes)

    # --- ResNet-50 ---
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    # --- ViT-B/16 ---
    # model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
    # model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

    return model

In [ ]:
MODEL_PATH = os.path.join(os.getcwd(), "model.pth")

def predict_test(device="cpu"):
    model = get_model()
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    model.to(device)
    print(f"Loaded model from {MODEL_PATH}")

    test_dir = os.path.join(DATA_DIR, "test")
    image_files = sorted(
        [f for f in os.listdir(test_dir) if f.endswith(".jpg")],
        key=lambda f: int(os.path.splitext(f)[0])
    )

    predictions = []
    with torch.no_grad():
        for fname in image_files:
            img = Image.open(os.path.join(test_dir, fname)).convert("RGB")
            x = test_transform(img).unsqueeze(0).to(device)
            pred = model(x).argmax(dim=1).item()
            predictions.append((fname, pred))

    out_path = os.path.join(os.getcwd(), "submission.csv")
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["ID", "Label"])
        writer.writerows(predictions)

    print(f"Saved {len(predictions)} predictions to {out_path}")
    return out_path

## The Training Pipeline (Fine-Tuning)

We fine-tune the entire pretrained ResNet-50 on our training data. The dataset is expanded 3x using augmented copies (flipped and rotated), and each copy also has on-the-fly random augmentations applied every epoch.

In [ ]:
from torch.utils.data import ConcatDataset, DataLoader, random_split

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load full dataset with train augmentation
full_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)

# 80/20 train/val split
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_subset, val_subset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# Augmented copies (flipped + rotated) — only for training portion
flip_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
full_flipped = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=flip_transform)
train_flipped, _ = random_split(full_flipped, [train_size, val_size], generator=torch.Generator().manual_seed(42))

rotate_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation((30, 90)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
full_rotated = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=rotate_transform)
train_rotated, _ = random_split(full_rotated, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_dataset = ConcatDataset([train_subset, train_flipped, train_rotated])

# Validation uses test_transform (no augmentation)
val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=test_transform)
_, val_dataset = random_split(val_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

print(f"Train: {len(train_dataset)} images (3x augmented from {train_size})")
print(f"Val: {len(val_dataset)} images")

model = get_model()
print(f"Model: ResNet-50, output classes: {NUM_CLASSES}")

## Hyper-parameters

| Parameter | Value |
|-----------|-------|
| Optimizer | SGD |
| Learning rate | 0.01 |
| Momentum | 0.9 |
| Weight decay | 1e-4 |
| Batch size | 32 |
| Max epochs | 30 |
| Early stopping patience | 5 |
| Validation split | 20% |
| Loss function | CrossEntropyLoss |
| Input size | 224x224 |

In [ ]:
import copy

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

num_epochs = 30
patience = 5

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")
best_model_state = None
epochs_no_improve = 0

for epoch in range(num_epochs):
    # --- Training ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)

    # --- Validation ---
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_acc = val_correct / val_total
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} — train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}")

    # --- Early stopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            break

# Restore best model
model.load_state_dict(best_model_state)
print(f"Restored best model with val_loss: {best_val_loss:.4f}")

In [ ]:
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

## Current Results, Training/Testing Curve

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history["train_loss"], label="Train Loss")
ax1.plot(epochs_range, history["val_loss"], label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training vs Validation Loss")
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_range, history["train_acc"], label="Train Acc")
ax2.plot(epochs_range, history["val_acc"], label="Val Acc")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Training vs Validation Accuracy")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
predict_test(device)

## What's Next?

- Add a validation split to track generalization
- Implement early stopping based on validation loss
- Try learning rate scheduling (e.g. cosine annealing)
- Experiment with other backbones (EfficientNet-B0, ViT-B/16)
- Freeze early backbone layers to prevent overfitting
- Test-time augmentation (TTA) for more robust predictions

In [ ]:
## What's Next?

- Try learning rate scheduling (e.g. cosine annealing)
- Experiment with other backbones (EfficientNet-B0, ViT-B/16)
- Freeze early backbone layers to prevent overfitting
- Test-time augmentation (TTA) for more robust predictions